# ML1 Homework — Revised Data & Feature Pipeline

## Notebook目的

将原课程项目的数据处理与特征工程改造成可审计的高频研究流程，重点修复非交易日填充、跨时段前值填充、时间顺序依赖和成交后信息进入特征的问题。

## 输入数据

- `02_Data/processed/features.pkl`：来源于已有数据处理流程的半秒级期货快照与基础特征。
- `02_Data/processed/trading_calendar.csv`：真实交易日历，至少包含 `trade_date`；如有 `is_trading_day` 字段，仅保留值为真的日期。

## 输出结果

- `Train_data_revised.pkl`
- `Validation_data_revised.pkl`
- `Test_data_revised.pkl`
- `feature_manifest_revised.csv`
- `data_quality_report_revised.json`

以上文件在运行后生成，不随本次代码迁移伪造。

## 对应研究文档

- [[../../02_Data/数据处理流程|数据处理流程]]
- [[../../03_Factors/高频特征工程|高频特征工程]]
- [[../../03_Factors/盘口结构特征|盘口结构特征]]
- [[../../01_Research/技术路线|技术路线]]


## 研究口径

- 决策时点为盘口快照时点 `t`。
- 预测目标为同一交易时段内未来3秒的 `last_price return`。
- 当前盘口字段可以用于时点 `t` 的信号；成交字段若用于特征，必须至少滞后一条500ms快照。
- 数据只在同一真实交易日、同一日内交易时段内补齐，禁止跨交易日或跨午休时段 `ffill`。
- 本Notebook不使用随机划分；Train、Validation、Test按日期先后划分。

当前版本只处理原研究实际使用的日盘。夜盘需要单独定义交易日归属后再加入。


In [ ]:
from pathlib import Path
from datetime import time
import json
import os
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "PROJECT_STATUS.md").exists():
            return candidate
    raise FileNotFoundError(
        "无法定位项目根目录。请从项目目录内运行，或设置 ML_FUTURES_PROJECT_ROOT。"
    )

PROJECT_ROOT = (
    Path(os.environ["ML_FUTURES_PROJECT_ROOT"]).resolve()
    if "ML_FUTURES_PROJECT_ROOT" in os.environ
    else find_project_root()
)
INPUT_PATH = PROJECT_ROOT / "02_Data/processed/features.pkl"
CALENDAR_PATH = PROJECT_ROOT / "02_Data/processed/trading_calendar.csv"
OUTPUT_DIR = PROJECT_ROOT / "02_Data/processed/revised"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_FREQ = pd.Timedelta("500ms")
HORIZON = pd.Timedelta(seconds=3)
HORIZON_STEPS = int(HORIZON / SNAPSHOT_FREQ)
MAX_FFILL_STEPS = 2  # 最多填充1秒，并且只允许在同一交易时段内发生
VALIDATION_DAYS = 2
TEST_DAYS = 2
COMPUTE_HURST = False  # 原实现计算量大，默认关闭；开启后仍严格按时段滚动

DAY_SESSIONS = {
    "day_am_1": (time(9, 0), time(10, 15)),
    "day_am_2": (time(10, 30), time(11, 30)),
    "day_pm": (time(13, 30), time(14, 55)),
}

REQUIRED_COLUMNS = {
    "trade_time", "instrumentid", "last_price", "avg_price", "vol",
    "pb1", "pa1", "vb1", "va1",
    "pb2", "pa2", "vb2", "va2",
    "pb3", "pa3", "vb3", "va3",
    "pb4", "pa4", "vb4", "va4",
    "pb5", "pa5", "vb5", "va5",
}


In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"缺少 {INPUT_PATH}。该文件应由已确认的上游数据处理流程生成。"
    )
if not CALENDAR_PATH.exists():
    raise FileNotFoundError(
        f"缺少 {CALENDAR_PATH}。为避免用工作日近似真实交易日，本版本要求提供真实交易日历。"
    )

data = pd.read_pickle(INPUT_PATH).copy()
calendar = pd.read_csv(CALENDAR_PATH)

missing_columns = sorted(REQUIRED_COLUMNS - set(data.columns))
if missing_columns:
    raise ValueError(f"输入数据缺少字段: {missing_columns}")

data["trade_time"] = pd.to_datetime(data["trade_time"], errors="raise")
calendar["trade_date"] = pd.to_datetime(
    calendar["trade_date"], errors="raise"
).dt.date
if "is_trading_day" in calendar.columns:
    calendar = calendar[calendar["is_trading_day"].astype(bool)]
valid_trade_dates = set(calendar["trade_date"])
if not valid_trade_dates:
    raise ValueError("交易日历中没有有效交易日。")


In [ ]:
def assign_day_session(timestamp):
    current_time = timestamp.time()
    for session_name, (start_time, end_time) in DAY_SESSIONS.items():
        if start_time <= current_time <= end_time:
            return session_name
    return None

def prepare_observed_data(frame):
    result = frame.copy()
    instrument_count = result["instrumentid"].nunique(dropna=True)
    if instrument_count != 1:
        raise ValueError(
            "当前研究流程要求输入单一合约，"
            f"但检测到 {instrument_count} 个instrumentid。"
        )
    result["trade_date"] = result["trade_time"].dt.date
    result = result[result["trade_date"].isin(valid_trade_dates)].copy()
    result["session_id"] = result["trade_time"].map(assign_day_session)
    result = result[result["session_id"].notna()].copy()
    result.sort_values("trade_time", inplace=True)

    duplicate_count = int(result.duplicated("trade_time").sum())
    if duplicate_count:
        duplicate_examples = result.loc[
            result.duplicated("trade_time", keep=False), "trade_time"
        ].head(10).tolist()
        raise ValueError(
            f"发现 {duplicate_count} 个重复时间戳，示例: {duplicate_examples}"
        )
    if not result["trade_time"].is_monotonic_increasing:
        raise ValueError("trade_time排序失败。")
    return result

observed = prepare_observed_data(data)


In [ ]:
def build_expected_index(trade_date, session_name, timezone):
    start_time, end_time = DAY_SESSIONS[session_name]
    start = pd.Timestamp.combine(trade_date, start_time)
    end = pd.Timestamp.combine(trade_date, end_time)
    if timezone is not None:
        start = start.tz_localize(timezone)
        end = end.tz_localize(timezone)
    return pd.date_range(start, end, freq=SNAPSHOT_FREQ)

def reindex_one_session(group):
    trade_date = group["trade_date"].iloc[0]
    session_name = group["session_id"].iloc[0]
    timezone = group["trade_time"].dt.tz
    expected = build_expected_index(trade_date, session_name, timezone)

    ordered = group.sort_values("trade_time").set_index("trade_time")
    ordered["_was_observed"] = True
    aligned = ordered.reindex(expected)
    aligned.index.name = "trade_time"
    aligned["_was_observed"] = aligned["_was_observed"].fillna(False)

    # 每个group只包含一个交易日和一个交易时段，因此不会跨日或跨时段填充。
    value_columns = [
        column for column in aligned.columns
        if column != "_was_observed"
    ]
    aligned[value_columns] = aligned[value_columns].ffill(
        limit=MAX_FFILL_STEPS
    )
    aligned["trade_date"] = trade_date
    aligned["session_id"] = session_name
    aligned["is_imputed"] = ~aligned["_was_observed"]
    aligned.drop(columns="_was_observed", inplace=True)
    aligned.reset_index(inplace=True)

    core_quote_columns = ["pb1", "pa1", "vb1", "va1"]
    aligned.dropna(subset=core_quote_columns, inplace=True)
    return aligned

aligned_sessions = []
for _, session_data in observed.groupby(
    ["trade_date", "session_id"], sort=True
):
    aligned_sessions.append(reindex_one_session(session_data))

aligned = pd.concat(aligned_sessions, ignore_index=True)
aligned.sort_values("trade_time", inplace=True)
aligned.reset_index(drop=True, inplace=True)


In [ ]:
def build_quality_report(frame):
    group_keys = ["trade_date", "session_id"]
    deltas = frame.groupby(group_keys)["trade_time"].diff()
    nonstandard_gaps = deltas.notna() & deltas.ne(SNAPSHOT_FREQ)

    crossed_book = frame["pb1"] > frame["pa1"]
    nonpositive_quotes = (frame[["pb1", "pa1"]] <= 0).any(axis=1)
    depth_columns = [
        f"{side}{level}"
        for level in range(1, 6)
        for side in ("vb", "va")
    ]
    negative_depth = (frame[depth_columns] < 0).any(axis=1)

    report = {
        "row_count": int(len(frame)),
        "trade_date_count": int(frame["trade_date"].nunique()),
        "duplicate_trade_time_count": int(
            frame.duplicated("trade_time").sum()
        ),
        "nonstandard_gap_count": int(nonstandard_gaps.sum()),
        "crossed_book_count": int(crossed_book.sum()),
        "nonpositive_quote_count": int(nonpositive_quotes.sum()),
        "negative_depth_count": int(negative_depth.sum()),
        "imputed_row_count": int(frame["is_imputed"].sum()),
        "imputed_row_ratio": float(frame["is_imputed"].mean()),
    }
    return report

quality_report = build_quality_report(aligned)
if quality_report["duplicate_trade_time_count"] != 0:
    raise ValueError("质量检查失败：仍存在重复时间戳。")
if quality_report["crossed_book_count"] != 0:
    raise ValueError("质量检查失败：存在买一价高于卖一价的交叉盘口。")
if quality_report["negative_depth_count"] != 0:
    raise ValueError("质量检查失败：存在负挂单量。")

quality_report


## 特征时间有效性处理

当前 `last_price` 不直接进入模型。原来的 `D_k`、`effective_spread` 和 `avg_price` 可能依赖当前快照已经发生的成交结果，因此新版只保留严格滞后一条快照的版本：

- `D_k_lag1`
- `effective_spread_lag1`
- `avg_price_lag1`

所有 `shift`、`diff` 和 `rolling` 都在“交易日 × 交易时段”内部执行。午休、跨日和不同交易时段之间不会共享历史值。


In [ ]:
GROUP_KEYS = ["trade_date", "session_id"]

def add_quote_and_depth_features(frame):
    result = frame.copy()
    grouped = result.groupby(GROUP_KEYS, sort=False)

    result["mid_price"] = (result["pb1"] + result["pa1"]) / 2
    result["spread"] = result["pa1"] - result["pb1"]
    result["relative_spread"] = result["spread"] / result["mid_price"]
    result["mid_return_1"] = grouped["mid_price"].pct_change(1)
    result["mid_return_6_lagged"] = grouped["mid_price"].pct_change(6)

    bid_columns = [f"vb{level}" for level in range(1, 6)]
    ask_columns = [f"va{level}" for level in range(1, 6)]
    result["bid_depth"] = result[bid_columns].sum(axis=1)
    result["ask_depth"] = result[ask_columns].sum(axis=1)
    depth_sum = result["bid_depth"] + result["ask_depth"]
    result["order_imbalance"] = np.where(
        depth_sum.ne(0),
        (result["bid_depth"] - result["ask_depth"]) / depth_sum,
        np.nan,
    )

    grouped = result.groupby(GROUP_KEYS, sort=False)
    for window in (10, 120):
        result[f"OI_MA_{window}"] = grouped[
            "order_imbalance"
        ].transform(
            lambda values: values.rolling(
                window, min_periods=window
            ).mean()
        )
        result[f"OI_momentum_{window}"] = grouped[
            "order_imbalance"
        ].diff(window)
        result[f"OI_std_{window}"] = grouped[
            "order_imbalance"
        ].transform(
            lambda values: values.rolling(
                window, min_periods=window
            ).std()
        )
    result["OI_skew_10"] = grouped["order_imbalance"].transform(
        lambda values: values.rolling(10, min_periods=10).skew()
    )

    pressure_denominator = (
        result["pb1"] * result["vb1"]
        + result["pa1"] * result["va1"]
    )
    result["market_pressure"] = np.where(
        pressure_denominator.ne(0),
        (
            result["pb1"] * result["vb1"]
            - result["pa1"] * result["va1"]
        )
        / pressure_denominator,
        np.nan,
    )

    for level in range(1, 6):
        denominator = result[f"vb{level}"] + result[f"va{level}"]
        result[f"SOIR{level}"] = np.where(
            denominator.ne(0),
            (
                result[f"vb{level}"] - result[f"va{level}"]
            )
            / denominator,
            np.nan,
        )
    result["SOIR_weighted"] = sum(
        (6 - level) * result[f"SOIR{level}"]
        for level in range(1, 6)
    )
    return result

features = add_quote_and_depth_features(aligned)


In [ ]:
def add_lagged_trade_features(frame):
    result = frame.copy()
    grouped = result.groupby(GROUP_KEYS, sort=False)

    previous_last = grouped["last_price"].shift(1)
    previous_bid = grouped["pb1"].shift(1)
    previous_ask = grouped["pa1"].shift(1)
    previous_mid = (previous_bid + previous_ask) / 2

    result["D_k_lag1"] = np.select(
        [
            previous_last > previous_mid,
            previous_last < previous_mid,
        ],
        [1.0, -1.0],
        default=0.0,
    )
    result.loc[previous_last.isna(), "D_k_lag1"] = np.nan
    result["effective_spread_lag1"] = (
        2
        * result["D_k_lag1"]
        * (previous_last - previous_mid)
        / previous_mid
    )
    result["avg_price_lag1"] = grouped["avg_price"].shift(1)
    result["avg_price_lag1_missing"] = (
        result["avg_price_lag1"].isna().astype(int)
    )
    return result

features = add_lagged_trade_features(features)


In [ ]:
def add_order_flow_features(frame):
    result = frame.copy()
    grouped = result.groupby(GROUP_KEYS, sort=False)

    for level in range(1, 6):
        bid_price = f"pb{level}"
        ask_price = f"pa{level}"
        bid_volume = f"vb{level}"
        ask_volume = f"va{level}"

        previous_bid_price = grouped[bid_price].shift(1)
        previous_ask_price = grouped[ask_price].shift(1)
        previous_bid_volume = grouped[bid_volume].shift(1)
        previous_ask_volume = grouped[ask_volume].shift(1)

        bid_flow = np.select(
            [
                result[bid_price] > previous_bid_price,
                result[bid_price] < previous_bid_price,
            ],
            [result[bid_volume], -previous_bid_volume],
            default=result[bid_volume] - previous_bid_volume,
        )
        ask_flow = np.select(
            [
                result[ask_price] < previous_ask_price,
                result[ask_price] > previous_ask_price,
            ],
            [result[ask_volume], -previous_ask_volume],
            default=result[ask_volume] - previous_ask_volume,
        )
        result[f"OFI{level}"] = bid_flow - ask_flow

    # 不同时保留MOFI和全部五档OFI，避免确定性重复。
    return result

features = add_order_flow_features(features)


In [ ]:
def hurst_exponent(values):
    values = np.asarray(values, dtype=float)
    scales = np.array([10, 20, 50, 100])
    rs_values = []
    valid_scales = []

    for scale in scales:
        segment_count = len(values) // scale
        if segment_count == 0:
            continue
        segment_rs = []
        for segment_index in range(segment_count):
            segment = values[
                segment_index * scale:(segment_index + 1) * scale
            ]
            standard_deviation = np.std(segment, ddof=1)
            if not np.isfinite(standard_deviation) or standard_deviation == 0:
                continue
            cumulative_deviation = np.cumsum(segment - np.mean(segment))
            segment_rs.append(
                (
                    np.max(cumulative_deviation)
                    - np.min(cumulative_deviation)
                )
                / standard_deviation
            )
        if segment_rs and np.mean(segment_rs) > 0:
            valid_scales.append(scale)
            rs_values.append(np.mean(segment_rs))

    if len(valid_scales) < 2:
        return np.nan
    return np.polyfit(
        np.log(valid_scales), np.log(rs_values), 1
    )[0]

if COMPUTE_HURST:
    grouped = features.groupby(GROUP_KEYS, sort=False)
    for level in range(1, 6):
        for side in ("vb", "va"):
            column = f"{side}{level}"
            output_column = f"hurst_{side}{level}"
            features[output_column] = grouped[column].transform(
                lambda values: values.rolling(
                    300, min_periods=300
                ).apply(hurst_exponent, raw=True)
            )


In [ ]:
BASE_FEATURE_COLUMNS = [
    "spread", "relative_spread", "mid_return_1", "mid_return_6_lagged",
    "bid_depth", "ask_depth", "order_imbalance",
    "OI_MA_10", "OI_MA_120",
    "OI_momentum_10", "OI_momentum_120",
    "OI_std_10", "OI_std_120", "OI_skew_10",
    "market_pressure",
    "SOIR1", "SOIR2", "SOIR3", "SOIR4", "SOIR5", "SOIR_weighted",
    "OFI1", "OFI2", "OFI3", "OFI4", "OFI5",
    "D_k_lag1", "effective_spread_lag1",
    "avg_price_lag1", "avg_price_lag1_missing",
    "is_imputed",
]
HURST_COLUMNS = [
    column for column in features.columns
    if column.startswith("hurst_")
]
FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + HURST_COLUMNS

grouped = features.groupby(GROUP_KEYS, sort=False)
features["future_trade_time"] = grouped["trade_time"].shift(
    -HORIZON_STEPS
)
features["future_last_price"] = grouped["last_price"].shift(
    -HORIZON_STEPS
)
exact_horizon = (
    features["future_trade_time"] - features["trade_time"]
).eq(HORIZON)
features["target_return_3s"] = np.where(
    exact_horizon,
    features["future_last_price"] / features["last_price"] - 1,
    np.nan,
)

MODEL_METADATA_COLUMNS = [
    "trade_time", "trade_date", "session_id", "instrumentid",
    "pb1", "pa1", "mid_price", "last_price",
]
model_data = features[
    MODEL_METADATA_COLUMNS + FEATURE_COLUMNS + ["target_return_3s"]
].copy()

infinite_count = int(
    np.isinf(model_data[FEATURE_COLUMNS].to_numpy(dtype=float)).sum()
)
if infinite_count:
    raise ValueError(f"特征中存在 {infinite_count} 个无穷值。")

# 不用0掩盖缺失原因；只保留具备完整特征和精确3秒标签的样本。
model_data.dropna(
    subset=FEATURE_COLUMNS + ["target_return_3s"], inplace=True
)
model_data.sort_values("trade_time", inplace=True)


In [ ]:
ordered_dates = sorted(model_data["trade_date"].unique())
minimum_date_count = VALIDATION_DAYS + TEST_DAYS + 1
if len(ordered_dates) < minimum_date_count:
    raise ValueError(
        f"有效交易日不足：至少需要 {minimum_date_count} 天，"
        f"当前只有 {len(ordered_dates)} 天。"
    )

test_dates = ordered_dates[-TEST_DAYS:]
validation_dates = ordered_dates[
    -(TEST_DAYS + VALIDATION_DAYS):-TEST_DAYS
]
train_dates = ordered_dates[:-(TEST_DAYS + VALIDATION_DAYS)]

train_data = model_data[
    model_data["trade_date"].isin(train_dates)
].copy()
validation_data = model_data[
    model_data["trade_date"].isin(validation_dates)
].copy()
test_data = model_data[
    model_data["trade_date"].isin(test_dates)
].copy()

assert train_data["trade_time"].max() < validation_data["trade_time"].min()
assert validation_data["trade_time"].max() < test_data["trade_time"].min()

train_data.to_pickle(OUTPUT_DIR / "Train_data_revised.pkl")
validation_data.to_pickle(OUTPUT_DIR / "Validation_data_revised.pkl")
test_data.to_pickle(OUTPUT_DIR / "Test_data_revised.pkl")

feature_manifest = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "uses_current_last_price": False,
        "availability": "decision_time_or_strictly_lagged",
    }
)
feature_manifest.to_csv(
    OUTPUT_DIR / "feature_manifest_revised.csv", index=False
)

quality_report.update(
    {
        "feature_count": len(FEATURE_COLUMNS),
        "model_row_count": int(len(model_data)),
        "train_dates": [str(value) for value in train_dates],
        "validation_dates": [str(value) for value in validation_dates],
        "test_dates": [str(value) for value in test_dates],
        "target": "future_3_second_last_price_return",
        "results_generated": False,
    }
)
(OUTPUT_DIR / "data_quality_report_revised.json").write_text(
    json.dumps(quality_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

{
    "train_shape": train_data.shape,
    "validation_shape": validation_data.shape,
    "test_shape": test_data.shape,
    "feature_count": len(FEATURE_COLUMNS),
}


## 运行后检查清单

- 确认真实交易日历覆盖全部样本日期。
- 检查 `data_quality_report_revised.json` 中的填充比例和异常计数。
- 人工确认 FastBox 的盘口字段和成交字段时间戳语义。
- 在进入模型训练前冻结数据版本与 `feature_manifest_revised.csv`。

本Notebook不生成模型结果，也不保证优化后收益提升。
